In [1]:
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

file_path = "../data/employee_attrition_engineered.xlsx"

df = pd.read_excel(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)

Dataset loaded successfully!
Shape: (1470, 42)


In [2]:
X = df.drop(columns=["Attrition", "Employee_ID"])

if df["Attrition"].dtype == "object":
    y = (
        df["Attrition"]
        .astype(str)
        .str.strip()
        .str.lower()
        .map({
            "no": 0,
            "yes": 1
        })
    )
else:
    y = df["Attrition"].astype(int)

print("X shape:", X.shape)
print("y shape:", y.shape)
print("Missing target values:", y.isna().sum())

X shape: (1470, 40)
y shape: (1470,)
Missing target values: 0


In [3]:
categorical_features = X.select_dtypes(
    include=["object", "category"]
).columns.tolist()

numerical_features = X.select_dtypes(
    include=["int64", "float64"]
).columns.tolist()

print("Categorical features:", len(categorical_features))
print("Numerical features:", len(numerical_features))

Categorical features: 7
Numerical features: 33


C:\Users\yashw\AppData\Local\Temp\ipykernel_28576\3828698519.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = X.select_dtypes(


In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing :", X_test.shape)

Training: (1176, 40)
Testing : (294, 40)


In [5]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                sparse_output=False
            ),
            categorical_features
        )
    ]
)

print("Preprocessor ready!")

Preprocessor ready!


In [6]:
negative_count = (y_train == 0).sum()
positive_count = (y_train == 1).sum()

scale_pos_weight = negative_count / positive_count

print("Negative:", negative_count)
print("Positive:", positive_count)
print("Scale Pos Weight:", round(scale_pos_weight, 2))

Negative: 986
Positive: 190
Scale Pos Weight: 5.19


In [7]:
logistic_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            LogisticRegression(
                class_weight="balanced",
                max_iter=2000,
                random_state=42
            )
        )
    ]
)

random_forest_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            RandomForestClassifier(
                n_estimators=300,
                class_weight="balanced",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

xgb_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        (
            "classifier",
            XGBClassifier(
                n_estimators=300,
                max_depth=4,
                learning_rate=0.05,
                subsample=0.8,
                colsample_bytree=0.8,
                scale_pos_weight=scale_pos_weight,
                objective="binary:logistic",
                eval_metric="logloss",
                random_state=42,
                n_jobs=-1
            )
        )
    ]
)

print("All models created successfully!")

All models created successfully!


In [8]:
best_model = xgb_model
best_model_name = "XGBoost"

In [9]:
best_model = random_forest_model
best_model_name = "Random Forest"

In [10]:
best_model = logistic_model
best_model_name = "Logistic Regression"

In [11]:
print("Final selected model:", best_model_name)

Final selected model: Logistic Regression


In [12]:
best_model.fit(X_train, y_train)

print(best_model_name, "trained successfully!")

Logistic Regression trained successfully!


In [13]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

y_pred_final = best_model.predict(X_test)
y_prob_final = best_model.predict_proba(X_test)[:, 1]

final_accuracy = accuracy_score(y_test, y_pred_final)
final_precision = precision_score(y_test, y_pred_final)
final_recall = recall_score(y_test, y_pred_final)
final_f1 = f1_score(y_test, y_pred_final)
final_auc = roc_auc_score(y_test, y_prob_final)

print("FINAL MODEL:", best_model_name)
print("--------------------------------")
print("Accuracy :", round(final_accuracy, 4))
print("Precision:", round(final_precision, 4))
print("Recall   :", round(final_recall, 4))
print("F1-Score :", round(final_f1, 4))
print("ROC-AUC  :", round(final_auc, 4))

FINAL MODEL: Logistic Regression
--------------------------------
Accuracy : 0.7585
Precision: 0.3636
Recall   : 0.6809
F1-Score : 0.4741
ROC-AUC  : 0.8147


In [14]:
df["Attrition_Probability"] = (
    best_model.predict_proba(X)[:, 1]
)

df["Attrition_Probability"] = (
    df["Attrition_Probability"].round(4)
)

df[
    [
        "Employee_ID",
        "Department",
        "JobRole",
        "Attrition",
        "Attrition_Probability"
    ]
].head(10)

,Employee_ID,Department,JobRole,Attrition,Attrition_Probability
0,EMP-0001,Sales,Sales Executive,1,0.9032
1,EMP-0002,Research & Development,Research Scientist,0,0.1177
2,EMP-0003,Research & Development,Laboratory Technician,1,0.8481
3,EMP-0004,Research & Development,Research Scientist,0,0.3102
4,EMP-0005,Research & Development,Laboratory Technician,0,0.7558
5,EMP-0006,Research & Development,Laboratory Technician,0,0.2090
6,EMP-0007,Research & Development,Laboratory Technician,0,0.5899
7,EMP-0008,Research & Development,Laboratory Technician,0,0.5067
8,EMP-0009,Research & Development,Manufacturing Director,0,0.3144
9,EMP-0010,Research & Development,Healthcare Representative,0,0.1389


In [15]:
def assign_risk(probability):
    if probability < 0.30:
        return "Low"
    elif probability <= 0.60:
        return "Medium"
    else:
        return "High"


df["Risk_Category"] = (
    df["Attrition_Probability"]
    .apply(assign_risk)
)

df[
    [
        "Employee_ID",
        "Attrition_Probability",
        "Risk_Category"
    ]
].head(10)

,Employee_ID,Attrition_Probability,Risk_Category
0,EMP-0001,0.9032,High
1,EMP-0002,0.1177,Low
2,EMP-0003,0.8481,High
3,EMP-0004,0.3102,Medium
4,EMP-0005,0.7558,High
5,EMP-0006,0.2090,Low
6,EMP-0007,0.5899,Medium
7,EMP-0008,0.5067,Medium
8,EMP-0009,0.3144,Medium
9,EMP-0010,0.1389,Low


In [16]:
risk_distribution = df["Risk_Category"].value_counts()

print(risk_distribution)

Risk_Category
Low       776
Medium    366
High      328
Name: count, dtype: int64


In [17]:
risk_percentage = (
    df["Risk_Category"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nRisk Distribution (%):")
print(risk_percentage)


Risk Distribution (%):
Risk_Category
Low       52.79
Medium    24.90
High      22.31
Name: proportion, dtype: float64


In [18]:
risk_summary = (
    df.groupby("Risk_Category")
    .agg(
        Employee_Count=("Employee_ID", "count"),
        Average_Probability=("Attrition_Probability", "mean")
    )
    .reset_index()
)

risk_summary["Average_Probability"] = (
    risk_summary["Average_Probability"]
    .round(4)
)

risk_summary

,Risk_Category,Employee_Count,Average_Probability
0,High,328,0.7898
1,Low,776,0.1194
2,Medium,366,0.4495


In [19]:
department_risk = (
    df.groupby("Department")
    .agg(
        Employees=("Employee_ID", "count"),
        Average_Risk=("Attrition_Probability", "mean"),
        High_Risk_Employees=(
            "Risk_Category",
            lambda x: (x == "High").sum()
        )
    )
    .reset_index()
)

department_risk["Average_Risk"] = (
    department_risk["Average_Risk"]
    .round(4)
)

department_risk

,Department,Employees,Average_Risk,High_Risk_Employees
0,Human Resources,63,0.3766,15
1,Research & Development,961,0.3062,173
2,Sales,446,0.4445,140


In [20]:
role_risk = (
    df.groupby(["Department", "JobRole"])
    .agg(
        Employees=("Employee_ID", "count"),
        Average_Risk=("Attrition_Probability", "mean"),
        High_Risk_Employees=(
            "Risk_Category",
            lambda x: (x == "High").sum()
        )
    )
    .reset_index()
)

role_risk["Average_Risk"] = (
    role_risk["Average_Risk"]
    .round(4)
)

role_risk.sort_values(
    "Average_Risk",
    ascending=False
).head(10)

,Department,JobRole,Employees,Average_Risk,High_Risk_Employees
10,Sales,Sales Representative,83,0.6862,52
3,Research & Development,Laboratory Technician,259,0.4925,99
0,Human Resources,Human Resources,52,0.4234,15
9,Sales,Sales Executive,326,0.4050,84
7,Research & Development,Research Scientist,292,0.3100,52
8,Sales,Manager,37,0.2505,4
5,Research & Development,Manufacturing Director,145,0.2325,11
2,Research & Development,Healthcare Representative,131,0.1948,9
4,Research & Development,Manager,54,0.1695,1
1,Human Resources,Manager,11,0.1555,0


In [21]:
model_path = "../models/final_attrition_model.pkl"

joblib.dump(
    best_model,
    model_path
)

print("Final model saved successfully!")
print(model_path)

Final model saved successfully!
../models/final_attrition_model.pkl


In [22]:
output_path = "../data/employee_attrition_scored.xlsx"

df.to_excel(
    output_path,
    index=False
)

print("Risk-scored dataset saved successfully!")
print(output_path)

Risk-scored dataset saved successfully!
../data/employee_attrition_scored.xlsx


In [23]:
loaded_model = joblib.load(
    "../models/final_attrition_model.pkl"
)

test_predictions = loaded_model.predict_proba(
    X_test
)[:5, 1]

print("Saved model loaded successfully!")
print("Test probabilities:")
print(test_predictions)

Saved model loaded successfully!
Test probabilities:
[0.46022232 0.01912771 0.32930137 0.03056805 0.82599837]
